In [1]:
import pandas as pd
import numpy as np

In [14]:
files = [
    "/kaggle/input/datasets/crazyapil/dataset/IT Support Ticket Data.csv",
    "/kaggle/input/datasets/crazyapil/dataset/aa_dataset_tickets_en_lang.csv",
    "/kaggle/input/datasets/crazyapil/dataset/customer_support_tickets_200k.csv"
]

In [15]:
dfs = []
for f in files:
  temp_df = pd.read_csv(f)
  if "body" in temp_df.columns:
    temp_df = temp_df["body"]
  elif "issue_description" in temp_df.columns:
    temp_df = temp_df["issue_description"]
  elif "Body" in temp_df.columns:
    temp_df = temp_df["Body"]
  dfs.append(temp_df)

In [16]:
df = pd.concat(dfs, ignore_index = True)

In [17]:
df = df.to_frame(name = "description")

In [18]:
df.head(5)

,description
0,"Dear Customer Support Team,I am writing to rep..."
1,"Dear Customer Support Team,I hope this message..."
2,"Dear Customer Support Team,I hope this message..."
3,"Dear Support Team,I hope this message reaches ..."
4,"Dear Customer Support,I hope this message reac..."


In [19]:
df = df.dropna()

In [20]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [21]:
def clean_text(text, keep_punctuation=True, lemmatize=False, remove_stopwords=False):
    """
    Enhanced text cleaning with configurable options
    """
    original_text = text
    
    text = text.lower()
    
    text = re.sub(r"^(dear|hi|hello|hey)\s+.*?,", "", text, flags=re.IGNORECASE)
    
    polite_phrases = [
        r"i hope this message finds you well",
        r"i hope this message reaches you",
        r"i hope you are doing well",
        r"i am reaching out to",
        r"just writing to",
        r"i wanted to",
        r"i would like to",
        r"i am writing to (report|inform|ask|request)",
        r"please (find|see) attached",
        r"attached please find"
    ]
    
    for phrase in polite_phrases:
        text = re.sub(phrase, "", text, flags=re.IGNORECASE)
    
    signoffs = [
        r"thank you.*$", r"thanks.*$", r"regards.*$", 
        r"best regards.*$", r"sincerely.*$", r"cheers.*$",
        r"appreciate your (help|assistance|time).*$",
        r"looking forward.*$"
    ]
    
    for signoff in signoffs:
        text = re.sub(signoff, "", text, flags=re.IGNORECASE)
    
    text = re.sub(r"\b\w+@\w+\.\w+\b", "", text)  # emails
    text = re.sub(r"\b\d{10,}\b", "", text)  # phone numbers
    text = re.sub(r"ticket\s*id:?\s*\w+", "", text, flags=re.IGNORECASE)  # ticket IDs
    
    if keep_punctuation:
        text = re.sub(r"[^\w\s\.\?\!]", "", text)
    else:
        text = re.sub(r"[^\w\s]", "", text)
    
    text = re.sub(r"\s+", " ", text).strip()
    
    if remove_stopwords or lemmatize:
        tokens = text.split()
        
        if remove_stopwords:
            stop_words = set(stopwords.words('english'))
            domain_words = {'not', 'no', 'but', 'however', 'issue', 'problem', 'error'}
            stop_words = stop_words - domain_words
            tokens = [word for word in tokens if word not in stop_words]
        
        if lemmatize:
            lemmatizer = WordNetLemmatizer()
            tokens = [lemmatizer.lemmatize(word) for word in tokens]
        
        text = ' '.join(tokens)
    
    if not text or len(text.split()) < 2:
        text = original_text.lower()
        text = re.sub(r"[^\w\s]", "", text)
        text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [22]:
df["cleaned_text"] = df["description"].apply(clean_text)

In [23]:
df = df.drop_duplicates(subset=['cleaned_text'], keep='first')

print(f"Original rows: {len(df)}")
print(f"After dedup on cleaned_text: {len(df)}")

Original rows: 25787
After dedup on cleaned_text: 25787


In [24]:
df["cleaned_text"]

0        a significant problem with the centralized acc...
1        well. request detailed information about the c...
2        . request clarification about the billing and ...
3        well. ask about the compatibility of your prod...
4        in good health. i am eager to learn more about...
                               ...                        
45994    there seems to be a discrepancy in my billing ...
45995              request a refund for the recent charge.
45996    twofactor authentication codes are not being d...
45998    i am unable to access my account after enterin...
46000    i am experiencing very slow performance while ...
Name: cleaned_text, Length: 25787, dtype: object

In [25]:
%%capture
!pip install sentence-transformers

In [26]:
from sentence_transformers import SentenceTransformer
import torch

In [27]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [37]:
model_name = "all-mpnet-base-v2"

In [38]:
embedder = SentenceTransformer(model_name)
embedder.to(device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'max_seq_length': 384, 'do_lower_case': False, 'architecture': 'MPNetModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [39]:
texts = df["cleaned_text"].tolist()

In [40]:
embeddings = embedder.encode(
    texts,
    batch_size = 64,
    show_progress_bar = True,
    normalize_embeddings = True
)

Batches:   0%|          | 0/403 [00:00<?, ?it/s]

In [46]:
%%capture
!pip install cudf-cu12 cuml-cu12 --extra-index-url=https://pypi.nvidia.com

In [47]:
import cuml.cluster

In [58]:
import cuml

umap_model = cuml.UMAP(
    n_components=5,       
    n_neighbors=15,       
    min_dist=0.0,        
    metric='cosine'      
)
reduced_embeddings = umap_model.fit_transform(embeddings)

hdb = cuml.cluster.HDBSCAN(
    min_cluster_size=30, 
    min_samples=1,        
    metric='euclidean',   
    cluster_selection_epsilon=0.1, 
    cluster_selection_method='eom'
)
labels_gpu = hdb.fit_predict(reduced_embeddings)

In [59]:
n_clusters = len(np.unique(labels_gpu)) - (1 if -1 in labels_gpu else 0)
print(f"GPU HDBSCAN: {n_clusters} clusters")

GPU HDBSCAN: 197 clusters


In [60]:
unique, counts = np.unique(labels_gpu, return_counts=True)
n_clusters = len(unique) - (1 if -1 in unique else 0)
noise_count = counts[unique == -1][0] if -1 in unique else 0

print(f"Total clusters: {n_clusters}")
print(f"Noise / outliers: {noise_count} ({noise_count / len(labels_gpu) * 100:.1f}%)")
print(f"Largest 10 clusters sizes: {sorted(counts[unique != -1], reverse=True)[:10]}")
print(f"Small clusters (<20 tickets): {sum(counts < 20)}")

Total clusters: 197
Noise / outliers: 3904 (15.1%)
Largest 10 clusters sizes: [np.int64(1512), np.int64(755), np.int64(723), np.int64(712), np.int64(665), np.int64(526), np.int64(513), np.int64(508), np.int64(433), np.int64(391)]
Small clusters (<20 tickets): 0


In [61]:
import cuml
import cupy as cp

is_clustered = labels_gpu != -1
is_noise = labels_gpu == -1

knn = cuml.neighbors.KNeighborsClassifier(n_neighbors=1)
knn.fit(embeddings[is_clustered], labels_gpu[is_clustered])

noise_reassignments = knn.predict(embeddings[is_noise])

final_labels = labels_gpu.copy()
final_labels[is_noise] = noise_reassignments

noise_count = cp.sum(final_labels == -1)
print(f"Remaining noise: {noise_count}")

Remaining noise: 0


In [63]:
from sklearn.feature_extraction.text import CountVectorizer

def get_cluster_keywords(texts, labels, n_words=10):
    df = pd.DataFrame({'text': texts, 'label': labels})
    docs_per_class = df.groupby(['label'], as_index=False).agg({'text': ' '.join})
    
    count = CountVectorizer(stop_words='english').fit(docs_per_class.text)
    t = count.transform(docs_per_class.text).toarray()
    w = t.sum(axis=1)
    tf = np.divide(t.T, w)
    sum_t = t.sum(axis=0)
    idf = np.log(np.divide(len(docs_per_class), sum_t, out=np.zeros_like(sum_t, dtype=float), where=sum_t!=0))
    tf_idf = np.multiply(tf.T, idf)
    
    words = count.get_feature_names_out()
    top_n_words = {label: [words[i] for i in tf_idf[i_label].argsort()[-n_words:]] 
                   for i_label, label in enumerate(docs_per_class.label)}
    return top_n_words

keywords = get_cluster_keywords(texts, final_labels)

In [64]:
keywords

{0: ['cost',
  'savings',
  'highpriority',
  'costeffectiveness',
  'instances',
  'ec2',
  'costeffective',
  'unnecessary',
  'instance',
  'expenses'],
 1: ['budgeting',
  'adjustment',
  'spike',
  'surge',
  'clarifying',
  'correspondence',
  'correction',
  'expenses',
  'months',
  'justify'],
 2: ['jam',
  'wirelessly',
  'paper',
  'print',
  'mobile',
  'duplex',
  'printing',
  'canon',
  'pixma',
  'mg3620'],
 3: ['printing',
  'jams',
  'print',
  'epson',
  'paper',
  'ecotank',
  'et4760',
  'hp',
  'deskjet',
  '3755'],
 4: ['internally',
  'onsite',
  'dropouts',
  'weeks',
  'technician',
  'drops',
  'routers',
  'networking',
  'enterprise',
  'isr4331'],
 5: ['screen',
  'oled',
  'xps',
  'c1',
  'black',
  '9310',
  'brightness',
  'flickers',
  'display',
  'flickering'],
 6: ['webinars',
  'screensharing',
  'sharing',
  'video',
  'discussions',
  'calls',
  'conferences',
  'conferencing',
  '11',
  'zoom'],
 7: ['flickering',
  'replacement',
  'ideapad',


In [66]:
import cupy as cp
import pandas as pd
from sklearn.cluster import AgglomerativeClustering


unique_labels = sorted([l for l in np.unique(final_labels) if l != -1])
centroids = cp.array([embeddings[final_labels == l].mean(axis=0) for l in unique_labels])

n_meta_clusters = 30 
merger = AgglomerativeClustering(n_clusters=n_meta_clusters, metric='cosine', linkage='average')
meta_labels = merger.fit_predict(centroids.get())

cluster_to_meta_map = {old_idx: meta_idx for old_idx, meta_idx in zip(unique_labels, meta_labels)}

df['cluster_id'] = final_labels
df['meta_cluster_id'] = df['cluster_id'].map(cluster_to_meta_map)

df['meta_label_name'] = df['meta_cluster_id'].apply(lambda x: f"Meta-Group {x}")

In [69]:
df.head(5)

,description,cleaned_text,cluster_id,meta_cluster_id,meta_label_name
0,"Dear Customer Support Team,I am writing to rep...",a significant problem with the centralized acc...,24,9,Meta-Group 9
1,"Dear Customer Support Team,I hope this message...",well. request detailed information about the c...,65,11,Meta-Group 11
2,"Dear Customer Support Team,I hope this message...",. request clarification about the billing and ...,110,4,Meta-Group 4
3,"Dear Support Team,I hope this message reaches ...",well. ask about the compatibility of your prod...,21,24,Meta-Group 24
4,"Dear Customer Support,I hope this message reac...",in good health. i am eager to learn more about...,97,24,Meta-Group 24


In [68]:
df['meta_label_name'].value_counts()

meta_label_name
Meta-Group 2     6451
Meta-Group 25    5563
Meta-Group 11    3163
Meta-Group 3     2345
Meta-Group 0     1789
Meta-Group 4     1385
Meta-Group 6     1366
Meta-Group 7      703
Meta-Group 27     363
Meta-Group 24     287
Meta-Group 9      244
Meta-Group 14     215
Meta-Group 26     209
Meta-Group 23     167
Meta-Group 18     159
Meta-Group 12     158
Meta-Group 16     129
Meta-Group 29     127
Meta-Group 8      126
Meta-Group 21     125
Meta-Group 28     120
Meta-Group 5       99
Meta-Group 13      97
Meta-Group 22      82
Meta-Group 19      71
Meta-Group 1       68
Meta-Group 10      65
Meta-Group 20      44
Meta-Group 17      36
Meta-Group 15      31
Name: count, dtype: int64

In [74]:
from collections import Counter

In [75]:
cluster_counts = Counter(labels_gpu)
sorted_clusters = sorted(cluster_counts.items(), key = lambda x: x[1], reverse = True)

In [79]:
cluster_counts = df["meta_cluster_id"].value_counts()
top_clusters = cluster_counts.index.sort_values() 

for cid in top_clusters:
    if cid == -1: continue 
    
    cluster_df = df[df["meta_cluster_id"] == cid]
    cluster_size = len(cluster_df)
    
    if cluster_size == 0:
        continue
        
    print(f"\n--- Meta-Group {cid} ({cluster_size} tickets) ---")
    
    samples = cluster_df["cleaned_text"].sample(
        min(5, cluster_size), 
        random_state=42
    )

    for i, text in enumerate(samples, 1):
        preview = str(text).replace('\n', ' ').strip()
        print(f"  {i}. {preview[:180]}...")


--- Meta-Group 0 (1789 tickets) ---
  1. i am writing to seek assistance in obtaining materials that detail the process of integrating scikitlearn with laravel 8 for the purpose of investment analysis. i am currently work...
  2. is it possible to offer guidelines for optimizing investment strategies using data analytics tools to assist in making informed decisions?...
  3. customer support seeking assistance to optimize data analytics for investment strategies at our firm. could you provide specifics on the tools and techniques used to analyze market...
  4. request assistance in optimizing our investment analytics dashboard. realtime data updates are essential to ensure accurate and timely decisionmaking and we also need enhanced secu...
  5. seeking to enhance investment tracking and integrate advanced data analytics for improved operational efficiency and better decisionmaking....

--- Meta-Group 1 (68 tickets) ---
  1. a significant issue with our database services impacting clie

In [ ]:
cluster_names = {
    'Meta-Group 0': 'Financial Data Analytics Support',
    'Meta-Group 1': 'System Outage & Service Disruption',
    'Meta-Group 2': 'General Technical Issues (Application & System Errors)',
    'Meta-Group 3': 'Documentation & Product Inquiries',
    'Meta-Group 4': 'Billing & Payment Issues',
    'Meta-Group 5': 'Third-Party Tools & Integration Requests',
    'Meta-Group 6': 'Service Downtime & Availability Issues',
    'Meta-Group 7': 'Application Functionality & Sync Issues',
    'Meta-Group 8': 'ClickUp Workflows & Integrations',
    'Meta-Group 9': 'Login & Access Issues',
    'Meta-Group 10': 'Security Software Integration & Setup',
    'Meta-Group 11': 'Digital Marketing Strategy & Performance',
    'Meta-Group 12': 'Collaboration Tools Connectivity & Hardware Issues',
    'Meta-Group 13': 'Email & Marketing Platform Integrations',
    'Meta-Group 14': 'Data & Analytics Platform Integrations Issues',
    'Meta-Group 15': 'IntelliJ IDEA Issues & Upgrades',
    'Meta-Group 16': 'Hardware Issues & Customer Support',
    'Meta-Group 17': 'AI & Analytics Tool Integrations',
    'Meta-Group 18': 'Jira Configuration & Support',
    'Meta-Group 19': 'General Support Requests',
    'Meta-Group 20': 'Redis Integration & Analytics',
    'Meta-Group 21': 'Network Connectivity & Router Issues',
    'Meta-Group 22': 'Zapier Integration & Workflow Automation',
    'Meta-Group 23': 'Printer Hardware & Connectivity Issues',
    'Meta-Group 24': 'Service & Subscription Information Requests',
    'Meta-Group 25': 'Data Security & Breach Mitigation',
    'Meta-Group 26': 'Consumer Hardware Issues & Purchase Support',
    'Meta-Group 27': 'Application & Software Stability Issues',
    'Meta-Group 28': 'Elasticsearch Integration & Analytics Optimization',
    'Meta-Group 29': 'Cloud & SaaS Integration Issues'
}

In [84]:
final_cluster_names = {
    'Meta-Group 0': 'Financial Data Analytics Support',
    'Meta-Group 1': 'Service Downtime & System Outages',
    'Meta-Group 2': 'Application & System Issues',
    'Meta-Group 3': 'Product & Documentation Inquiries',
    'Meta-Group 4': 'Billing & Payment Issues',
    'Meta-Group 5': 'Third-Party Tool Integrations',
    'Meta-Group 6': 'Service Downtime & System Outages',       # duplicate of 1
    'Meta-Group 7': 'Application Functionality & Sync Issues',
    'Meta-Group 8': 'Software & Platform Support',
    'Meta-Group 9': 'Login & Access Issues',
    'Meta-Group 10': 'Security Software Integration & Setup',
    'Meta-Group 11': 'Digital Marketing Strategy & Performance',
    'Meta-Group 12': 'Collaboration Tools Connectivity & Hardware Issues',
    'Meta-Group 13': 'Third-Party Tool Integrations',          # merged into 5
    'Meta-Group 14': 'Third-Party Tool Integrations',          # merged into 5
    'Meta-Group 15': 'Software & Platform Support',            # merged into 8
    'Meta-Group 16': 'Consumer Hardware Issues & Purchase Support',
    'Meta-Group 17': 'Third-Party Tool Integrations',          # merged into 5
    'Meta-Group 18': 'Software & Platform Support',            # merged into 8
    'Meta-Group 19': 'General Support Requests',
    'Meta-Group 20': 'Third-Party Tool Integrations',          # merged into 5
    'Meta-Group 21': 'Network Connectivity & Router Issues',
    'Meta-Group 22': 'Third-Party Tool Integrations',          # merged into 5
    'Meta-Group 23': 'Consumer Hardware Issues & Purchase Support', # merged into 16
    'Meta-Group 24': 'Billing & Payment Issues',               # merged into 4
    'Meta-Group 25': 'Security Software Integration & Setup',  # merged into 10
    'Meta-Group 26': 'Consumer Hardware Issues & Purchase Support', # merged into 16
    'Meta-Group 27': 'Application & System Issues',            # merged into 2
    'Meta-Group 28': 'Third-Party Tool Integrations',          # merged into 5
    'Meta-Group 29': 'Third-Party Tool Integrations'           # merged into 5
}

In [87]:
df["merged_category"] = df["meta_label_name"].map(final_cluster_names)

df["merged_category"] = df["merged_category"].fillna("Uncategorized / Rare Issues")

print(df["merged_category"].value_counts())

merged_category
Application & System Issues                           6814
Security Software Integration & Setup                 5628
Digital Marketing Strategy & Performance              3163
Product & Documentation Inquiries                     2345
Financial Data Analytics Support                      1789
Billing & Payment Issues                              1672
Service Downtime & System Outages                     1434
Third-Party Tool Integrations                          820
Application Functionality & Sync Issues                703
Consumer Hardware Issues & Purchase Support            505
Software & Platform Support                            316
Login & Access Issues                                  244
Collaboration Tools Connectivity & Hardware Issues     158
Network Connectivity & Router Issues                   125
General Support Requests                                71
Name: count, dtype: int64


In [88]:
import json
with open("merged_cluster_names.json", "w") as f:
    json.dump(final_cluster_names, f, indent=2)

In [90]:
import pickle

with open('umap_model.pkl', 'wb') as f:
    pickle.dump(umap_model, f)

with open('knn_model.pkl', 'wb') as f:
    pickle.dump(knn, f)

with open('category_mapping.pkl', 'wb') as f:
    pickle.dump(final_cluster_names, f)

In [99]:
import re

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = " ".join(text.split())
    return text

def classify_new_ticket(ticket_text):
    clean_text = preprocess_text(ticket_text)
    
    new_embedding = embedder.encode([clean_text], normalize_embeddings=True)

    raw_pred = knn.predict(new_embedding)
    cluster_id = int(raw_pred[0]) 
    
    meta_id = cluster_to_meta_map.get(cluster_id, -1)
    key = f"Meta-Group {meta_id}"
    category = final_cluster_names.get(key, "Uncategorized")
    
    return category

In [104]:
clean = "Our latest campaign on Facebook is underperforming and the CPC is way too high. We need to re-evaluate the targeting."
print(f"Clean Result: {classify_new_ticket(clean)}")

Clean Result: Digital Marketing Strategy & Performance


In [105]:
import joblib

In [106]:
joblib.dump(embeddings, 'best_embedding.joblib')

['best_embedding.joblib']

In [109]:
embedder.save('/kaggle/working/')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [110]:
joblib.dump(knn, '/kaggle/working/knn_classifier.joblib')

['/kaggle/working/knn_classifier.joblib']

In [112]:
mapping_bundle = {
    'cluster_to_meta': cluster_to_meta_map,
    'meta_to_name': final_cluster_names
}
with open('/kaggle/working/mapping_logic.pkl', 'wb') as f:
    pickle.dump(mapping_bundle, f)